# Mapping Review (Project 3, Steps 4, 7 and 8)

The structural matcher (`compare_structures.py --shape coarse --normalize families --presence-only`) only generates **candidates**: two classes match when their restrictions have the same *shape* (for example "only existential restrictions"), whatever properties and fillers they use. This notebook records a decision and a reason for **every** candidate row in the `*-structural-matches-with-defs.xlsx` files, and checks that the mapping TTLs contain exactly what was accepted.

Decision values:
* `accept (asserted)`: the axiom is in a `*-mapping.ttl` file.
* `accept (entailed)`: true, but not asserted row by row. A more general mapping plus the ontologies' own axioms entail it (shown in `reasoner-report.xlsx`).
* `reject`: no subclass/equivalence relation holds; the reason says why.

Output: `src/data/mapping-review.xlsx` (one sheet per pair, plus a sheet of mappings found only through the definition review).

In [1]:
from pathlib import Path
import re
import pandas as pd
from rdflib import Graph, URIRef, RDFS, OWL

NB_DIR = Path.cwd().resolve()
SRC_DIR = NB_DIR.parent / "src"
DATA_DIR = SRC_DIR / "data"

def mapping_axioms(*names):
    g = Graph()
    for n in names:
        g.parse(DATA_DIR / n)
    out = {}
    for s, p, o in g:
        if p == OWL.equivalentClass:
            out[(str(s), str(o))] = out[(str(o), str(s))] = "equivalentClass"
        elif p == RDFS.subClassOf:
            out[(str(s), str(o))] = "subClassOf"
            out.setdefault((str(o), str(s)), "superClassOf")
    return out

def ancestors(g: Graph, iri: str) -> set:
    return {str(a) for a in g.transitive_objects(URIRef(iri), RDFS.subClassOf)}

def load(pair):
    return pd.read_excel(DATA_DIR / f"{pair}-structural-matches-with-defs.xlsx")

BFO = "http://purl.obolibrary.org/obo/BFO_"
review = {}

## BFO – IES

Enrichment gave four IES classes existential restrictions (Event, EventParticipant, State, ParticularPeriod) and five BFO classes too (process, history, SDC, realizable entity, GDC). The matcher pairs every "existential only" class on one side with every one on the other, giving 16 rows. None of them is a true subclass or equivalence pair. The real BFO–IES correspondences came from reading the definitions side by side (step 7) and are listed separately.

In [2]:
REASONS_BFO_IES = {
    ("history", "Event"): "BFO history = the whole-life sum of processes in the region a material entity occupies; an IES Event is one bounded activity or incident. A history is not an event, and no event is a whole-life history.",
    ("history", "State"): "Closest match in this set: IES treats an entity's whole life as a State of itself, which resembles a BFO history. But IES State also covers every temporal slice, and IES Entity (e.g. Person) is also a State, so asserting it would force BFO processes and IES entities together. Close match only; not asserted.",
    ("history", "Event Participant"): "An EventParticipant is the state of one entity during one event, not a whole-life history.",
    ("history", "Particular Period"): "History is a process; ParticularPeriod is a stretch of time (BFO temporal region). BFO keeps processes and temporal regions disjoint.",
}
def bfo_ies_reason(l, r):
    if (l, r) in REASONS_BFO_IES:
        return REASONS_BFO_IES[(l, r)]
    return (f"Category mismatch: '{l}' is a BFO dependent continuant (it exists by depending on a bearer), "
            f"while IES '{r}' is a spatio-temporal Element (an extent in space-time). The only shared feature is that "
            f"both have an existential restriction.")

df = load("bfo-core-ies")
maps = mapping_axioms("bfo-mapping.ttl", "ies-mapping.ttl")
df["decision"] = ["accept (asserted)" if (a, b) in maps else "reject" for a, b in zip(df.left_iri, df.right_iri)]
df["relation"] = [maps.get((a, b), "") for a, b in zip(df.left_iri, df.right_iri)]
df["reason"] = [bfo_ies_reason(l, r) for l, r in zip(df.left_label, df.right_label)]
review["bfo-core-ies"] = df
df[["left_label", "right_label", "decision", "reason"]]

,left_label,right_label,decision,reason
0,generically dependent continuant,State,reject,Category mismatch: 'generically dependent cont...
1,generically dependent continuant,Particular Period,reject,Category mismatch: 'generically dependent cont...
2,generically dependent continuant,Event,reject,Category mismatch: 'generically dependent cont...
3,generically dependent continuant,Event Participant,reject,Category mismatch: 'generically dependent cont...
4,realizable entity,State,reject,Category mismatch: 'realizable entity' is a BF...
5,realizable entity,Particular Period,reject,Category mismatch: 'realizable entity' is a BF...
6,realizable entity,Event,reject,Category mismatch: 'realizable entity' is a BF...
7,realizable entity,Event Participant,reject,Category mismatch: 'realizable entity' is a BF...
8,history,State,reject,Closest match in this set: IES treats an entit...
9,history,Particular Period,reject,History is a process; ParticularPeriod is a st...


## CCOM – QUDT

After enrichment, every unit class on both sides says which quantity kind its units measure (`quantityKind value qk:X`). The matcher cannot see the filler, so it pairs all of them (828 rows). A row is accepted when both classes name the **same quantity kind** and the textual definitions agree. The relation (equivalence vs subclass) is taken from `ccom-mapping.ttl`.

In [3]:
qk = lambda ax: set(re.findall(r"quantityKind value (qk:\w+)", str(ax)))
df = load("ccom-qudt")
maps = mapping_axioms("ccom-mapping.ttl", "qudt-mapping.ttl")
decisions, relations, reasons = [], [], []
for _, row in df.iterrows():
    L, R = qk(row.left_axioms), qk(row.right_axioms)
    shared = sorted(L & R)
    rel = maps.get((row.left_iri, row.right_iri), "")
    if shared and rel:
        decisions.append("accept (asserted)"); relations.append(rel)
        reasons.append(f"Same quantity kind {', '.join(shared)}; definitions agree"
                       + ("" if rel == "equivalentClass" else " (the QUDT class also covers other quantity kinds)"))
    elif shared:
        decisions.append("REVIEW"); relations.append(""); reasons.append("Shared quantity kind but no mapping axiom")
    else:
        decisions.append("reject"); relations.append("")
        reasons.append(f"Different quantity kinds ({', '.join(sorted(L)) or 'none'} vs {', '.join(sorted(R)) or 'none'})"
                       if R else "QUDT class has no quantity-kind axiom (a grouping class matched only on its typePrefix restriction)")
df["decision"], df["relation"], df["reason"] = decisions, relations, reasons
review["ccom-qudt"] = df
assert not (df.decision == "REVIEW").any()
print(df.decision.value_counts().to_string())
df[df.decision != "reject"][["left_label", "relation", "right_label", "reason"]]

decision
reject               805
accept (asserted)     23


,left_label,relation,right_label,reason
9,Measurement Unit of Rotational Inertia,equivalentClass,Angular Mass Unit,Same quantity kind qk:MomentOfInertia; definit...
42,Measurement Unit of Pressure,subClassOf,Pressure Or Stress Unit,Same quantity kind qk:Pressure; definitions ag...
80,Measurement Unit of Length,equivalentClass,Length Unit,Same quantity kind qk:Length; definitions agree
115,Measurement Unit of Volume,equivalentClass,Volume Unit,Same quantity kind qk:Volume; definitions agree
176,Measurement Unit of Energy,subClassOf,Energy And Work Unit,Same quantity kind qk:Energy; definitions agre...
193,Measurement Unit of Mass Flow Rate,equivalentClass,Mass Per Time Unit,Same quantity kind qk:MassFlowRate; definition...
243,Measurement Unit of Acceleration,equivalentClass,Linear Acceleration Unit,Same quantity kind qk:LinearAcceleration; defi...
276,Measurement Unit of Density,equivalentClass,Mass Per Volume Unit,Same quantity kind qk:Density; definitions agree
322,Measurement Unit of Torque,subClassOf,Bending Moment Or Torque Unit,Same quantity kind qk:Torque; definitions agre...
349,Measurement Unit of Momentum,subClassOf,Linear Momentum Unit,Same quantity kind qk:LinearMomentum; definiti...


## CCOT – OWL-Time

OWL-Time splits time itself (`TemporalEntity`: `Instant`, `Interval`, `ProperInterval`, `DateTimeInterval`) from *information about* time (`TemporalPosition`, `TimePosition`, `DateTimeDescription`, `MonthOfYear`, `Duration`). CCOT classes are all BFO temporal regions, i.e. time itself. So:
* CCOT intervals match `ProperInterval`. This is entailed from one BFO-level mapping, not asserted per class.
* Gregorian Day/Year match `DateTimeInterval` (asserted).
* Everything paired with a description, position or duration class is a category error.
* CCOT instants paired with `ProperInterval` contradict OWL-Time's disjointness of instants and proper intervals.

In [4]:
g_ccot = Graph(); g_ccot.parse(SRC_DIR / "ccot.ttl"); g_ccot.parse(SRC_DIR / "bfo-core.ttl")
g_to = Graph(); g_to.parse(SRC_DIR / "time.ttl")
TIME = "http://www.w3.org/2006/time#"
INFO = {"TemporalPosition", "TimePosition", "GeneralDateTimeDescription", "DateTimeDescription",
        "MonthOfYear", "January", "Year", "Duration", "TemporalDuration", "DurationDescription", "GeneralDurationDescription"}
df = load("ccot-time")
maps = mapping_axioms("ccot-mapping.ttl", "to-mapping.ttl")
decisions, relations, reasons = [], [], []
for _, row in df.iterrows():
    anc = ancestors(g_ccot, row.left_iri)
    r_local = row.right_iri.replace(TIME, "")
    deprecated = (URIRef(row.right_iri), OWL.deprecated, None) in g_to
    is_instant = BFO + "0000203" in anc
    is_interval = BFO + "0000202" in anc
    is_1d = BFO + "0000038" in anc and not is_interval
    rel = maps.get((row.left_iri, row.right_iri), "")
    if rel:
        decisions.append("accept (asserted)"); relations.append(rel)
        reasons.append("Gregorian calendar-aligned interval = an interval named by a Gregorian DateTimeDescription")
    elif r_local in INFO:
        decisions.append("reject"); relations.append("")
        reasons.append(f"Category error: time:{r_local} is information about time (a description, position or duration); "
                       f"'{row.left_label}' is a temporal region (time itself)" + ("; also deprecated in OWL-Time" if deprecated else ""))
    elif r_local == "ProperInterval" and is_interval:
        decisions.append("accept (entailed)"); relations.append("subClassOf")
        reasons.append("A BFO temporal interval has distinct begin and end; entailed by ccot-mapping 'temporal interval SubClassOf ProperInterval'")
    elif r_local == "ProperInterval" and is_1d:
        decisions.append("accept (entailed)"); relations.append("subClassOf")
        reasons.append("Asserted only as a 1D temporal region, but CCOT's own 'interval contains' domain makes it a temporal interval (see reasoner report), so the ProperInterval mapping follows")
    elif r_local == "ProperInterval" and is_instant:
        decisions.append("reject"); relations.append("")
        reasons.append("An instant has zero extent; time:Instant and time:ProperInterval are disjoint in OWL-Time")
    elif r_local == "DateTimeInterval":
        decisions.append("reject"); relations.append("")
        reasons.append("Instant, or not aligned to Gregorian calendar units (DateTimeDescription is Gregorian only)" if is_instant or "Julian" in row.left_label
                       else "Not necessarily calendar-aligned (e.g. any 24-hour Day, not a calendar day)")
    else:
        decisions.append("REVIEW"); relations.append(""); reasons.append("")
df["decision"], df["relation"], df["reason"] = decisions, relations, reasons
review["ccot-time"] = df
assert not (df.decision == "REVIEW").any()
print(df.decision.value_counts().to_string())
df[df.decision != "reject"][["left_label", "relation", "right_label", "decision"]]

decision
reject               133
accept (entailed)     17
accept (asserted)      2


,left_label,relation,right_label,decision
7,Multi-Hour Temporal Interval,subClassOf,Proper interval,accept (entailed)
15,Second,subClassOf,Proper interval,accept (entailed)
31,Minute,subClassOf,Proper interval,accept (entailed)
33,Gregorian Year,subClassOf,Date-time interval,accept (asserted)
39,Gregorian Year,subClassOf,Proper interval,accept (entailed)
47,Week,subClassOf,Proper interval,accept (entailed)
55,Multi-Week Temporal Interval,subClassOf,Proper interval,accept (entailed)
63,Multi-Day Temporal Interval,subClassOf,Proper interval,accept (entailed)
71,Multi-Minute Temporal Interval,subClassOf,Proper interval,accept (entailed)
79,Multi-Year Temporal Interval,subClassOf,Proper interval,accept (entailed)


## Mappings found only through the definition review

These mappings are in the mapping files but have **no** structural candidate row. The structural pass missed them because the two classes have different restriction shapes: BFO `process` carries both `some` and `only` restrictions, while IES `Event` carries only `some`. Some classes, like `time:Instant`, have no restrictions at all.

In [5]:
extra = pd.DataFrame([
    ("bfo-ies", "ParticularPeriod", "subClassOf (both directions -> equivalent)", "temporal interval",
     "IES: 'a specific, contiguous extent of time'; BFO: 'one-dimensional temporal region that is continuous, without gaps'"),
    ("bfo-ies", "PeriodOfTime", "subClassOf", "temporal region",
     "IES: 'spatial extent everywhere, temporal extent limited' = a stretch of time; BFO temporal regions also include instants, so only subClassOf"),
    ("bfo-ies", "Event", "subClassOf", "process",
     "IES: 'activity or incident, involving one or more participating entities'; BFO process: occurrent with temporal parts and a material participant"),
    ("ccot-to", "temporal instant", "subClassOf (both directions -> equivalent)", "time:Instant",
     "BFO: 'zero-dimensional temporal region'; OWL-Time: 'temporal entity with zero extent or duration'"),
    ("ccot-to", "temporal interval", "subClassOf", "time:ProperInterval",
     "Continuous and non-degenerate, so begin and end are different"),
    ("ccot-to", "time:ProperInterval", "subClassOf", "one-dimensional temporal region",
     "Non-zero extent; OWL-Time does not say it is gap-free, so not mapped to 'temporal interval'"),
    ("ccot-to", "time:DateTimeInterval", "subClassOf", "temporal interval",
     "A single calendar unit (a given day, month or year) is contiguous"),
    ("ccot-to", "time:TemporalEntity", "subClassOf", "temporal region", "Instants and intervals are both temporal regions"),
    ("ccom-qudt", "QUDT broader unit classes", "subClassOf", "CCO Measurement Unit",
     "EnergyAndWork, PressureOrStress, BendingMomentOrTorque, LinearMomentum, LinearVelocity units are measurement units"),
], columns=["pair", "subject", "relation", "object", "justification"])

with pd.ExcelWriter(DATA_DIR / "mapping-review.xlsx") as xw:
    for pair, df in review.items():
        cols = ["decision", "relation", "left_label", "right_label", "reason", "left_definition", "right_definition", "left_iri", "right_iri"]
        df[cols].sort_values(["decision", "left_label"]).to_excel(xw, sheet_name=pair, index=False)
    extra.to_excel(xw, sheet_name="definition-review", index=False)
summary = pd.DataFrame([{"pair": p, "candidates": len(d), **d.decision.value_counts().to_dict()} for p, d in review.items()]).fillna(0)
print("Wrote", DATA_DIR / "mapping-review.xlsx")
summary

Wrote C:\Users\luaya\UB_applied_Onto\Ontology-Tradecraft\projects\project-3\assignment\src\data\mapping-review.xlsx


,pair,candidates,reject,accept (asserted),accept (entailed)
0,bfo-core-ies,16,16,0.0,0.0
1,ccom-qudt,828,805,23.0,0.0
2,ccot-time,152,133,2.0,17.0
